# Перенос и синхронизация весов из GCP Cloud Storage на Google Drive (Бесплатно)

Данный ноутбук использует утилиту **rsync** для синхронизации весов моделей из Google Cloud Storage (`gs://...`) на ваш Google Drive.

### Преимущества rsync:
- **Бесплатный трафик**: Передача данных происходит во внутренней сети Google.
- **Докачка (Идемпотентность)**: В случае обрыва Colab повторный запуск скачает только недостающие или частично загруженные файлы.

### Шаг 1: Подключение Google Drive

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
print("Google Drive успешно примонтирован.")

Mounted at /content/drive
Google Drive успешно примонтирован.


### Шаг 2: Авторизация в Google Cloud (GCP)

In [2]:
from google.colab import auth

auth.authenticate_user()
print("Авторизация в GCP выполнена успешно.")

Авторизация в GCP выполнена успешно.


### Шаг 3: Настройка путей (GCS BUCKET -> Google Drive)

In [3]:
# Путь к файлам в GCS (например: gs://bebladii-weigths/checkpoints/)
GCS_SRC_PATH = "gs://bebladii-weigths-us/planB/phase3/checkpoints/"

# Папка сохранения на Google Drive
GDRIVE_DEST_DIR = "/content/drive/MyDrive/AI/BEBLaDII/Plan B/phase 3/weights_phase3_attempt20260726/"

os.makedirs(GDRIVE_DEST_DIR, exist_ok=True)
print(f"Источник (GCS): {GCS_SRC_PATH}")
print(f"Назначение (Drive): {GDRIVE_DEST_DIR}")

Источник (GCS): gs://bebladii-weigths-us/planB/phase3/checkpoints/
Назначение (Drive): /content/drive/MyDrive/AI/BEBLaDII/Plan B/phase 3/weights_phase3_attempt20260725/


### Шаг 4: Синхронизация весов (rsync)
Используем `gcloud storage rsync -r` для рекурсивного скачивания с функцией возобновления.

In [4]:
!gcloud config set component_manager/disable_update_check true
!gcloud config set core/verbosity error

Updated property [component_manager/disable_update_check].


To take a quick anonymous survey, run:
  $ gcloud survey

Updated property [core/verbosity].


In [5]:
# Основная команда через современный gcloud storage
# !gcloud storage rsync -r --threads=1 --processes=1 "{GCS_SRC_PATH}"  "{GDRIVE_DEST_DIR}"

# Запасная команда через gsutil с многопоточностью (-m):
# Однопоточная последовательная синхронизация (без перегрузки Google Drive FUSE)
# Флаг -o 'GSUtil:parallel_process_count=1' качает файлы СТРОГО по одному по очереди.
!gsutil -o "GSUtil:parallel_process_count=1" -o "GSUtil:parallel_thread_count=1" rsync -r "{GCS_SRC_PATH}" "{GDRIVE_DEST_DIR}"

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.

both the source and destination. Your crcmod installation isn't using the
module's C extension, so checksumming will run very slowly. If this is your
first rsync since updating gsutil, this rsync can take significantly longer than
usual. For help installing the extension, please see "gsutil help crcmod".

Building synchronization state...
Starting synchronization...
Copying gs://bebladii-weigths-us/planB/phase3/checkpoints/gcs_test.txt...
Copying gs://bebladii-weigths-us/planB/phase3/checkpoints/phase3_step_1995.pth...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster if you enabled sliced object downloads. This
feature is enabled by default but requires that compiled crcmod be
installed

### Шаг 5: Проверка результатов

In [6]:
print("Содержимое целевой папки на Google Drive:")
!ls -lh "{GDRIVE_DEST_DIR}"

Содержимое целевой папки на Google Drive:
total 62G
-rw------- 1 root root   24 Jul 25 20:15 gcs_test.txt
-rw------- 1 root root 4.4G Jul 25 23:27 phase3_step_1995_opt.pth
-rw------- 1 root root 4.4G Jul 25 23:27 phase3_step_1995.pth
-rw------- 1 root root 4.4G Jul 26 01:02 phase3_step_2995_opt.pth
-rw------- 1 root root 4.4G Jul 26 01:01 phase3_step_2995.pth
-rw------- 1 root root 4.4G Jul 26 02:37 phase3_step_3995_opt.pth
-rw------- 1 root root 4.4G Jul 26 02:36 phase3_step_3995.pth
-rw------- 1 root root 4.4G Jul 26 04:12 phase3_step_4995_opt.pth
-rw------- 1 root root 4.4G Jul 26 04:11 phase3_step_4995.pth
-rw------- 1 root root 4.4G Jul 26 05:47 phase3_step_5995_opt.pth
-rw------- 1 root root 4.4G Jul 26 05:46 phase3_step_5995.pth
-rw------- 1 root root 4.4G Jul 26 07:22 phase3_step_6995_opt.pth
-rw------- 1 root root 4.4G Jul 26 07:21 phase3_step_6995.pth
-rw------- 1 root root 4.4G Jul 25 21:53 phase3_step_995_opt.pth
-rw------- 1 root root 4.4G Jul 25 21:52 phase3_step_995.pth
